# Issue Index Generator

Generate a JSON index file for all issues in a given provider's OCR collection.
This tool discovers all issues in a provider directory and creates a structured index
organized by media_alias → year → month → issue records.

The output index can be used as input file for the impresso text import pipeline.

## Import Required Libraries

In [1]:
import json
import os
import re
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple
import logging
from tqdm import tqdm
from datetime import date
import pandas as pd
from ast import literal_eval
from tqdm import tqdm

# Register tqdm with pandas to enable progress_apply
tqdm.pandas()

## Configuration

Configure the provider collection parameters and output settings.
This changes for each provider's specific data situation

## SUB case

In [2]:
# Provider name and media aliases mapping
PROVIDER_NAME = "SUB"  # e.g., BNL, Olive, RERO, etc.

# Configuration parameters
BASE_DATA_PATH = f"/mnt/project_impresso/original"  # Update with actual provider root path

# Output configuration
OUTPUT_FILE = f"/home/piconti/impresso-text-acquisition/text_preparation/data/issue_indices/issues_index.{PROVIDER_NAME.lower()}.json"

# SUB uses directory structure-based discovery, not filename patterns
# Directory structure: [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]/[METS files]
# Use discover_issues_sub() function below, not the generic pattern-based discovery

print(f"Provider path: {BASE_DATA_PATH}")
print(f"Output file: {OUTPUT_FILE}")
print(f"Using SUB-specific directory structure discovery")

Provider path: /mnt/project_impresso/original
Output file: /home/piconti/impresso-text-acquisition/text_preparation/data/issue_indices/issues_index.sub.json
Using SUB-specific directory structure discovery


### Load Provider Collection Data

Discover all OCR files in the provider directory structure.

In [3]:
dir_to_alias_sub = {
    "Hamburger_Echo": "hamb_echo"
}

#### Define a sorting lambda function to assign edition letters 

In [4]:
# Edition sorting key: splits on '-', sorts by number first, then by edition type
# Returns (number, edition_priority) tuple
sort_edition = lambda e: (
    int(e.split('-')[0][1:]) if '-' in e and e[0] in ['A', 'M'] else 999,
    {"Ausgabe": -1, "Morgenausgabe": 0, "Abendausgabe": 1}.get(e.split('-')[-1] if '-' in e else e, 2)
)

# Test
test_editions = ["Ausgabe", "M1-Morgenausgabe", "A2-Abendausgabe", "M3-Morgenausgabe", "A1-Abendausgabe"]
print("Sorting with lambda:")
all_day_editions = sorted(test_editions, key=sort_edition)
for edition in test_editions:
    edition_l = chr(96 + all_day_editions.index(edition)+1)
    print(f"  {edition} → {sort_edition(edition)} --> {edition_l}")

Sorting with lambda:
  Ausgabe → (999, -1) --> e
  M1-Morgenausgabe → (1, 0) --> a
  A2-Abendausgabe → (2, 1) --> c
  M3-Morgenausgabe → (3, 0) --> d
  A1-Abendausgabe → (1, 1) --> b


Ensure we are going through all the data and extracting the correct info

In [6]:
Path('/mnt/project_impresso/original/SUB/Hamburger_Echo')

PosixPath('/mnt/project_impresso/original/SUB/Hamburger_Echo')

In [ ]:
for filepath in Path('/mnt/project_impresso/original/SUB/Hamburger_Echo').rglob("*/PPN*.xml"):
    print(f"filepath: {filepath}")

In [15]:
filepath = "/mnt/project_impresso/original/SUB/Hamburger_Echo/1900/04/27/Ausgabe/PPN1754726119_19000427.xml"

issue_dir_path = os.path.dirname(filepath).replace(BASE_DATA_PATH, "")
parts = issue_dir_path.rstrip("/").split("/")
year = parts[-4]
month = parts[-3]
day = parts[-2]
edition_name = parts[-1].split('/')[0]

print(issue_dir_path, parts)
year, month, day, edition_name

/SUB/Hamburger_Echo/1900/04/27/Ausgabe ['', 'SUB', 'Hamburger_Echo', '1900', '04', '27', 'Ausgabe']


/SUB/Hamburger_Echo/1900/04/27/Ausgabe ['', 'SUB', 'Hamburger_Echo', '1900', '04', '27', 'Ausgabe']


('1900', '04', '27', 'Ausgabe')

In [17]:
year, month, day = parts[-4:-1]
year, month, day

('1900', '04', '27')

#### define a list of image format extensions

In [6]:
# sort the image formats from most desirable to least
possible_img_formats = [".jp2", ".tif", ".tiff", ".jpg", ".jpeg", ".pdf", ".png"]

### debug the logic to identify the extension and image subdir

In [32]:
prov_base_dir = os.path.join(BASE_DATA_PATH, PROVIDER_NAME)

#eg_dir = "/mnt/project_impresso/original/SUB/Hamburger_Echo/1900/04/27/Ausgabe/PPN1754726119_19000427.xml"
eg_dir = "/mnt/project_impresso/original/SUB/Hamburger_Echo/1923/05/25/A2-Abendausgabe/PPN1754726119_19230525A2.xml"

issue_dir_path = os.path.dirname(eg_dir)
parts = issue_dir_path.rstrip("/").split("/")
year, month, day = parts[-4:-1]
edition_name = parts[-1].split('/')[0]

print(f"edition_name: {edition_name}")

sort_editions = lambda e: (
    int(e.split('-')[0][1:]) if '-' in e and e[0] in ['A', 'M'] else 999,
    {"Ausgabe": -1, "Morgenausgabe": 0, "Abendausgabe": 1}.get(e.split('-')[-1] if '-' in e else e, 2)
)
# For directories without explicit edition prefix
# This will be resolved later when grouping by day
day_dir = '/'.join(parts[:-1])
all_day_editions = sorted([d for d in os.listdir(day_dir) if os.path.isdir(os.path.join(day_dir, d))], key=sort_editions)

print(f"all_day_editions: {all_day_editions}")
edition = chr(96 + all_day_editions.index(edition_name)+1)

print(f"edition: {edition}")

non_xml_files = {'': [f for f in os.listdir(issue_dir_path) if not f.endswith('.xml')]}
subdirs = [d for d in os.listdir(issue_dir_path) if os.path.isdir(os.path.join(prov_base_dir, d))]

print(f"edinon_xml_filestion: {non_xml_files}")
print(f"subdirs: {subdirs}")

if len(non_xml_files) == 0:
    # there are no other files than xmls check if there are subdirs which contain some
    for subdir in subdirs:
        non_xml_files[subdir] = [f for f in os.listdir(os.path.join(issue_dir_path, subdir)) if not f.endswith('.xml')]
        print(f"non_xml_files: {non_xml_files}")

imgs_dir = None
img_ext = None
num_xml_files = len(list(Path(issue_dir_path).rglob('*.xml')))
for ext in possible_img_formats:
    if not imgs_dir and not img_ext:
        print(f"ext: {ext}")
        print(f"sum(ext in f for f in non_xml_files): {sum(ext in f for f in non_xml_files)+1}")
        print(f"len(Path(issue_dir_path).rglob('*.xml'))+1: {len(list(Path(issue_dir_path).rglob('*.xml')))}")
        for subdir, files in non_xml_files.items():
            files_with_ext = [f for f in files if ext in f]
            if len(files_with_ext)+1 == num_xml_files:
                # the extensions are sorted, the first one to match is the one of choice
                imgs_dir = subdir
                img_ext = ext
                break
            else:
                print(f"subdir {subdir} and extension {ext} - found {len(files_with_ext)} image files, but there are {num_xml_files} xml files")

print(f"img_ext: {img_ext}, imgs_dir: {imgs_dir}")

edition_name: A2-Abendausgabe
all_day_editions: ['A1-Abendausgabe', 'A2-Abendausgabe']
edition: b
edinon_xml_filestion: {'': ['00000002.tif', '00000001.tif']}
subdirs: []
ext: .jp2
sum(ext in f for f in non_xml_files): 1
len(Path(issue_dir_path).rglob('*.xml'))+1: 3
subdir  and extension .jp2 - found 0 image files, but there are 3 xml files
ext: .tif
sum(ext in f for f in non_xml_files): 1
len(Path(issue_dir_path).rglob('*.xml'))+1: 3
img_ext: .tif, imgs_dir: 


In [50]:
filepath = "/mnt/project_impresso/original/SUB/Hamburger_Echo/1900/04/27/Ausgabe/PPN1754726119_19000427.xml"
filepath.rstrip("/").split("/")[-5:-2], date(int(filepath.rstrip("/").split("/")[-5]), int(filepath.rstrip("/").split("/")[-4]), int(filepath.rstrip("/").split("/")[-3]))

(['1900', '04', '27'], datetime.date(1900, 4, 27))

In [13]:
Path(issue_dir_path).rglob('*.xml')

<generator object Path.rglob at 0x7f8b5f5579a0>

### aggregate all in a function which creates and writes to disk the issue index

In [ ]:
def detect_issues_sub(out_path, base_dir: str = BASE_DATA_PATH, prov: str = PROVIDER_NAME, dir_to_alias_sub=dir_to_alias_sub, write_file:bool=True, debug:bool=False) -> List[Dict]:
    """
    Discover SUB newspaper issues using the directory structure:
    [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]
    
    Issues are identified by the presence of METS XML files containing "PPN".
    
    Args:
        root_path: Root directory of the SUB collection
        
    Returns:
        List of issue metadata dictionaries with keys:
        {alias, year, month, day, edition, path, mets_file}
    """
    # First, list alias directories in base_dir for the provider
    prov_base_dir = os.path.join(base_dir, prov)

    try:
        title_dirs = [d for d in os.listdir(prov_base_dir) if os.path.isdir(os.path.join(prov_base_dir, d))]
    except OSError as e:
        print(f"Failed to list base directory {prov_base_dir}: {e}")
        return []
    
    all_issues = {}

    for title_dirname in title_dirs:

        # identify which to which alias this alias corresponds
        alias = dir_to_alias_sub[title_dirname]
        alias_issues = {}

        all_mets_files = [str(p) for p in Path(os.path.join(prov_base_dir, title_dirname)).rglob("*/PPN*.xml")]
        print(f"Found {len(all_mets_files)} mets files for {alias}, sorting them by date...")
        # sort all issue files to write years and days in the correct order
        sorted_mets = sorted(all_mets_files, key=lambda x: date(int(x.rstrip("/").split("/")[-5]), int(x.rstrip("/").split("/")[-4]), int(x.rstrip("/").split("/")[-3])))

        print(f"Done sorting the mets files by date, now starting creating the index...")

        # look for all the mets files, which have a "PPN" in their filename -> this means the issue dir is also found
        for filepath in tqdm(sorted_mets):
            
            # extract the relative path to the issue
            issue_dir_path = os.path.dirname(filepath)
            parts = issue_dir_path.rstrip("/").split("/")
            year, month, day = parts[-4:-1]
            edition_name = parts[-1].split('/')[0]

            if len([f for f in os.listdir(issue_dir_path) if f.endswith('.xml')])==1:
                print(f"Warning! There is only one xml file in dir {issue_dir_path}, skipping this issue")
                continue

            sort_editions = lambda e: (
                int(e.split('-')[0][1:]) if '-' in e and e[0] in ['A', 'M'] else 999,
                {"Ausgabe": -1, "Morgenausgabe": 0, "Abendausgabe": 1}.get(e.split('-')[-1] if '-' in e else e, 2)
            )
            # For directories without explicit edition prefix
            # This will be resolved later when grouping by day
            day_dir = '/'.join(parts[:-1])
            all_day_editions = sorted([d for d in os.listdir(day_dir) if os.path.isdir(os.path.join(day_dir, d))], key=sort_editions)
            edition = chr(96 + all_day_editions.index(edition_name)+1)
            

            # IDENTIFY THE SUBDIR OF THE IMAGES
            non_xml_files = {'': [f for f in os.listdir(issue_dir_path) if not f.endswith('.xml')]}
            subdirs = [d for d in os.listdir(issue_dir_path) if os.path.isdir(os.path.join(prov_base_dir, d))]

            #if len(non_xml_files['']) == 0:
            # there are no other files than xmls check if there are subdirs which contain some
            for subdir in subdirs:
                non_xml_files[subdir] = [f for f in os.listdir(os.path.join(issue_dir_path, subdir)) if not f.endswith('.xml')]

            # IDENTIFY THE EXTENSION OF THE IMAGES FILES
            imgs_dir = []
            img_ext = []
            num_xml_files = len(list(Path(issue_dir_path).rglob('*.xml')))
            for subdir, files in non_xml_files.items():
                #if not imgs_dir and not img_ext:
                    #print(f"ext: {ext}")
                    #print(f"sum(ext in f for f in non_xml_files): {sum(ext in f for f in non_xml_files)+1}")
                    #print(f"len(Path(issue_dir_path).rglob('*.xml'))+1: {len(list(Path(issue_dir_path).rglob('*.xml')))}")
                for ext in possible_img_formats:
                    files_with_ext = [f for f in files if ext in f]
                    if len(files_with_ext)+1 == num_xml_files:
                        # the extensions are sorted, the first one to match is the one of choice
                        imgs_dir.append(subdir)
                        img_ext.append(ext)
                        break
                    elif debug:
                        print(f"subdir {subdir} and extension {ext} - found {len(files_with_ext)} image files, but there are {num_xml_files} xml files")

            if len(imgs_dir)>1 or len(img_ext)>1:
                print(f"WARNING!! THERE MIGHT BE MULTIPLE IMAGE FORMATS; THE CHOSEN ONE WILL BE {img_ext[0]} PRESENT IN SUBDIR {imgs_dir[0]}, BUT THERE NEEDS TO BE AN XML CHECK!!")

            # sabe the metadata for this issue
            issue_meta_info = {
                "day": day,
                "edition": edition,
                "local_path": issue_dir_path.replace(base_dir, ""),
                "imgs_subdir": imgs_dir[0],
                "imgs_ext": img_ext[0]
            }

            if debug:
                print(f"Adding issue {alias}-{year}-{month}-{day}-{edition} to the index: {issue_meta_info}")

            # save the created metadata into the list of issues for this alias
            if year not in alias_issues:
                alias_issues[year] = {month: [issue_meta_info]}
            elif month not in alias_issues[year]:
                alias_issues[year][month] = [issue_meta_info]
            else:
                alias_issues[year][month].append(issue_meta_info)

        print(f"Adding {len(alias_issues)} issues for alias {alias} to the total list of aliases")
        all_issues[alias] = alias_issues

    if write_file:
        with open(out_path, "w") as fout:
            json.dump(all_issues, fout, indent=4)

    return all_issues

In [54]:
out_sub_issue_index_path = "/home/piconti/impresso-text-acquisition/text_preparation/data/issue_indices/issue_index.sub.json"

all_sub_issues = detect_issues_sub(out_sub_issue_index_path)

Found 15418 mets files for hamb_echo, sorting them by date...
Done sorting the mets files by date, now starting creating the index...


 62%|██████▏   | 9577/15418 [02:05<01:03, 91.89it/s]

Warning! There is only one xml file in dir /mnt/project_impresso/original/SUB/Hamburger_Echo/1919/01/24/Abendausgabe, skipping this issue


 63%|██████▎   | 9737/15418 [02:07<01:00, 94.44it/s]

Warning! There is only one xml file in dir /mnt/project_impresso/original/SUB/Hamburger_Echo/1919/05/03/Abendausgabe, skipping this issue


...

Warning! There is only one xml file in dir /mnt/project_impresso/original/SUB/Hamburger_Echo/1924/02/19/Ausgabe, skipping this issue
Warning! There is only one xml file in dir /mnt/project_impresso/original/SUB/Hamburger_Echo/1924/02/20/Ausgabe, skipping this issue
Warning! There is only one xml file in dir /mnt/project_impresso/original/SUB/Hamburger_Echo/1924/02/21/Ausgabe, skipping this issue


 84%|████████▎ | 12904/15418 [02:45<00:30, 81.60it/s]

Warning! There is only one xml file in dir /mnt/project_impresso/original/SUB/Hamburger_Echo/1925/10/03/Ausgabe, skipping this issue


100%|██████████| 15418/15418 [03:24<00:00, 75.36it/s]


Adding 47 issues for alias hamb_echo to the total list of aliases


## RTS Case

In [2]:


# Provider name and media aliases mapping
PROVIDER_NAME = "RTS"  # e.g., BNL, Olive, RERO, etc.

# Configuration parameters
BASE_DATA_PATH = f"/mnt/project_impresso/original"  # Update with actual provider root path

# Output configuration
OUTPUT_FILE = f"/home/piconti/impresso-text-acquisition/text_preparation/data/issue_indices/issues_index.{PROVIDER_NAME.lower()}.json"

# SUB uses directory structure-based discovery, not filename patterns
# Directory structure: [base]/[alias]/[yyyy]/[mm]/[dd]/[edition_name]/[METS files]
# Use discover_issues_sub() function below, not the generic pattern-based discovery

print(f"Provider path: {BASE_DATA_PATH}")
print(f"Output file: {OUTPUT_FILE}")
print(f"Using SUB-specific directory structure discovery")

Provider path: /mnt/project_impresso/original
Output file: /home/piconti/impresso-text-acquisition/text_preparation/data/issue_indices/issues_index.rts.json
Using SUB-specific directory structure discovery


### Read the processed metadata file, and filter our shows without audio files 

(also write this filtered version to disk)

In [11]:
rts_metadata_filepath = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/programs_metadata_rts.csv"

rts_metadata_df = pd.read_csv(rts_metadata_filepath)
print(len(rts_metadata_df))
rts_metadata_df.head()

26261


,alias,date_str,stt_filename,mp3_filenames,stripped_OID,exact_date,broadcast_date,cls_ID,OID,login,...,person_descriptors,thematical_descriptors,rights_usage_possibilities,radio_channels,subdomains,recording_dates,first_broadcast_dates,supports,spt_filenames,participants
0,ana_media,16/12/1996,677E6735-8DEA-44F1-A52F-142BFEF9AB2E_STT.xml,['677e6735-8dea-44f1-a52f-142bfef9ab2e_7UBM_05...,677E6735-8DEA-44F1-A52F-142BFEF9AB2E,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{677E6735-8DEA-44F1-A52F-142BFEF9AB2E},NaN,...,"['X-Files', 'Aux frontières du réel']","['série télèvisée', 'science-fiction']",NaN,['Espace 2'],['Interview'],['~__/12/1996 - ~__/12/1996'],['16/12/1996 - 16/12/1996'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,"[None, '7UBM_057338_1{8cfcf300-a815-4f92-bea2-...","[{'name': 'Frias, Roxanne', 'function': 'Inter..."
1,ana_media,26/05/1997,640E3C0B-40F0-4BD9-83C6-1FF4BCE16484_STT.xml,['640e3c0b-40f0-4bd9-83c6-1ff4bce16484_1BBM_05...,640E3C0B-40F0-4BD9-83C6-1FF4BCE16484,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{640E3C0B-40F0-4BD9-83C6-1FF4BCE16484},NaN,...,NaN,"['école primaire', 'Internet', 'technique péda...",NaN,['Espace 2'],"['Commentaire', 'Parlé divers']",['~__/05/1997 - ~__/05/1997'],['26/05/1997 - 26/05/1997'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['1BBM_058066_6{2782ba11-7050-44d3-bef1-a81252...,"[{'name': 'Dubois, Laurent', 'function': 'Inte..."
2,ana_media,20/01/1997,2AF13E50-C6A2-49E5-9851-06C441833B50_STT.xml,['2af13e50-c6a2-49e5-9851-06c441833b50_RPBM_05...,2AF13E50-C6A2-49E5-9851-06C441833B50,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{2AF13E50-C6A2-49E5-9851-06C441833B50},NaN,...,"['SSR', 'TSR']","['censure', 'télévision', 'morale']",NaN,['Espace 2'],['Interview'],['~__/04/1997 - ~__/04/1997'],['20/01/1997 - 20/01/1997'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,"[None, 'RPBM_057474_1{8116d0d3-1c12-44cf-a8a2-...","[{'name': 'Duparc, Nicole', 'function': 'Inter..."
3,ana_media,11/11/1996,C940BD19-5503-4918-852B-D40C1C18B359_STT.xml,['c940bd19-5503-4918-852b-d40c1c18b359_O1BM_05...,C940BD19-5503-4918-852B-D40C1C18B359,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{C940BD19-5503-4918-852B-D40C1C18B359},NaN,...,NaN,"['série télèvisée', 'urgence médicale', 'télés...",NaN,['Espace 2'],['Interview'],['~__/11/1996 - ~__/11/1996'],['11/11/1996 - 11/11/1996'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['O1BM_057153_2{396b576c-7ed9-4018-a251-c12a84...,"[{'name': 'Kiefer, B.', 'function': 'Interview..."
4,ana_media,21/04/1997,6EE0054C-042D-42E6-B11F-FEB6C0353B5A_STT.xml,['6ee0054c-042d-42e6-b11f-feb6c0353b5a_FHBM_05...,6EE0054C-042D-42E6-B11F-FEB6C0353B5A,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{6EE0054C-042D-42E6-B11F-FEB6C0353B5A},NaN,...,NaN,"['enseignement secondaire', 'technique pédagog...",NaN,['Espace 2'],['Interview'],['~__/04/1997 - ~__/04/1997'],['21/04/1997 - 21/04/1997'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['FHBM_055791_2{e8455981-078d-44f2-bb94-2e22cd...,"[{'name': 'Prophan, Geneviève', 'function': 'I..."


First remove the unused columns in this context.
A further filtering/processing of the metadata csv might be useful, but currently we're mainly aiming to collect the necessary info for the importer.

In [12]:
rts_metadata_df.columns

Index(['alias', 'date_str', 'stt_filename', 'mp3_filenames', 'stripped_OID',
       'exact_date', 'broadcast_date', 'cls_ID', 'OID', 'login',
       'broadcast_episode_title', 'sequence', 'broadcast_program_name',
       'document_type', 'hierarchy_level', 'modified_by', 'modified_on',
       'physical_support_history', 'production_type', 'recording_place',
       'rights_notes', 'rights_status', 'series_title', 'content_summary',
       'workflow_status', 'assembly_status', 'live', 'modulation_type',
       'work_duration', 'work_duration_compl', 'geographical_descriptors',
       'person_descriptors', 'thematical_descriptors',
       'rights_usage_possibilities', 'radio_channels', 'subdomains',
       'recording_dates', 'first_broadcast_dates', 'supports', 'spt_filenames',
       'participants'],
      dtype='str')

#### Check the cases where there is no valid file

for each of these files, we will try to find if there is any file which matches the OID

In [12]:
last_alias = None
for row in no_existing_mp3s.itertuples():
    print(row.alias, row.stripped_OID, row.spt_filenames, row.mp3_filenames)
    if row.alias!=last_alias:
        # update the values if we changed alias
        audios_dir = os.path.join(BASE_DATA_PATH, PROVIDER_NAME, row.alias, "audio")
        stt_dir = os.path.join(BASE_DATA_PATH, PROVIDER_NAME, row.alias, "stt")
        last_alias = row.alias
        print(f"Changed values of audios_dir to {audios_dir} and stt_dir to {stt_dir}. Listing their contents")
        all_audios = os.listdir(audios_dir)
        all_stt = os.listdir(stt_dir)

    #matching_stt = row.stripped_OID
    stt_file = f"{row.stripped_OID}_STT.xml"
    print(f"{row.alias}: {stt_file in all_stt} - stt_file={stt_file}")

    support_filenames = [f for f in literal_eval(row.spt_filenames) if f]
    all_valid_audios = []
    for f in support_filenames:
        if f and '.wav' in f:
            print(f"removing the .wav in {f}")
            f = f.replace(".wav", '')
        valid_audios = [audio for audio in all_audios if f in audio]
        print(f"filename {f} corresponds to audio {valid_audios}!")
        all_valid_audios.extend(valid_audios)
    
    oid_also_in = [audio for audio in all_valid_audios if row.stripped_OID.lower() in audio]
    if len(all_valid_audios)>1:
        print(f"{row.alias}: There are more than 1 audio file which matches: {all_valid_audios}")
    elif len(all_valid_audios)==1:
        print(f"{row.alias}: audio_file={all_valid_audios[0]}")
    else:
        print(f"{row.alias}: No valid audio file")

j_mat 074AD35B-19AE-4F31-8BAB-7D6A61967489 ['TZCDR_884_01_09{5847ce3d-8476-488c-9d45-eba06614fd65}', 'CDR_884_pl__09{56625d53-1078-4402-bdea-8983a06bd1af}', None] ['074ad35b-19ae-4f31-8bab-7d6a61967489_CDR_884_pl__09{56625d53-1078-4402-bdea-8983a06bd1mp3', '074ad35b-19ae-4f31-8bab-7d6a61967489_TZCDR_884_01_09{5847ce3d-8476-488c-9d45-eba06614fdmp3']
Changed values of audios_dir to /mnt/project_impresso/original/RTS/j_mat/audio and stt_dir to /mnt/project_impresso/original/RTS/j_mat/stt. Listing their contents
j_mat: False - stt_file=074AD35B-19AE-4F31-8BAB-7D6A61967489_STT.xml
filename TZCDR_884_01_09{5847ce3d-8476-488c-9d45-eba06614fd65} corresponds to audio ['074ad35b-19ae-4f31-8bab-7d6a61967489_TZCDR_884_01_09{5847ce3d-8476-488c-9d45-eba06614fd65}.mp3']!
filename CDR_884_pl__09{56625d53-1078-4402-bdea-8983a06bd1af} corresponds to audio ['074ad35b-19ae-4f31-8bab-7d6a61967489_CDR_884_pl__09{56625d53-1078-4402-bdea-8983a06bd1af}.mp3']!
j_mat: There are more than 1 audio file which match

## KBR case

In [ ]:
# Provider name and media aliases mapping
PROVIDER_NAME = "KBR"  # e.g., BNL, Olive, RERO, etc.

# Configuration parameters
BASE_DATA_PATH = f"/mnt/project_impresso/original/{PROVIDER_NAME}"  # Update with actual provider root path

# Output configuration
OUTPUT_FILE = f"/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/{PROVIDER_NAME}/issues_index.json"

# Pattern for extracting metadata from filenames
# Adjust these patterns based on your provider's naming convention
# Example pattern: 1924587_newspaper_actionfem_1927-10-15_01
FILENAME_PATTERN = r'.*?_newspaper_([a-z]+)_(\d{4})-(\d{2})-(\d{2})_(\d{2})'

# Example: For Olive format with dates in directory names
# FILENAME_PATTERN = r'(\d{4})/(\d{2})/(\d{2})/.*'

print(f"Provider path: {BASE_DATA_PATH}")
print(f"Output file: {OUTPUT_FILE}")
print(f"Pattern: {FILENAME_PATTERN}")

## Validate Index Output

Verify the generated JSON file is valid and complete.

In [ ]:
def validate_index(index_file: str) -> bool:
    """
    Validate the generated index file.
    
    Args:
        index_file: Path to the index JSON file
        
    Returns:
        True if valid, False otherwise
    """
    try:
        # Check file exists
        if not Path(index_file).exists():
            print(f"✗ Index file not found: {index_file}")
            return False
        
        # Load and parse JSON
        with open(index_file, 'r', encoding='utf-8') as f:
            loaded_index = json.load(f)
        
        # Count statistics
        total_media = len(loaded_index)
        total_years = sum(len(years) for years in loaded_index.values())
        total_months = sum(
            len(months) 
            for years in loaded_index.values() 
            for months in years.values()
        )
        
        # Count issues based on structure type
        if STRUCTURE_TYPE == "array":
            total_issues = sum(
                len(issues) 
                for years in loaded_index.values()
                for months in years.values()
                for issues in months.values()
            )
        else:
            total_issues = sum(
                len(issues) 
                for years in loaded_index.values()
                for months in years.values()
                for issues in months.values()
            )
        
        print("✓ Index file validation results:")
        print(f"  Media aliases: {total_media}")
        print(f"  Years: {total_years}")
        print(f"  Months: {total_months}")
        print(f"  Issues: {total_issues}")
        
        # Show breakdown by media alias
        print("\n  Breakdown by media:")
        for alias in sorted(loaded_index.keys()):
            years = loaded_index[alias]
            year_ranges = [f"{min(years.keys())}-{max(years.keys())}"]
            issue_count = sum(
                len(months) for months in years.values()
            )
            print(f"    {alias}: {len(years)} years ({', '.join(year_ranges)}), {issue_count} issues")
        
        return True
        
    except json.JSONDecodeError as e:
        print(f"✗ Invalid JSON: {e}")
        return False
    except Exception as e:
        print(f"✗ Validation error: {e}")
        return False


# Validate the output
if Path(OUTPUT_FILE).exists():
    is_valid = validate_index(OUTPUT_FILE)
    if is_valid:
        print("\n✓ Index file is valid and ready for use!")
    else:
        print("\n✗ Index file validation failed")
else:
    print(f"✗ Output file was not created at {OUTPUT_FILE}")